In [1]:
%%capture
!pip install lancedb

In [2]:
import json
from sentence_transformers import SentenceTransformer
import lancedb

In [3]:
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

<All keys matched successfully>


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [4]:
jd_schema = {'role_title': 'Senior AI Engineer — Founding Team', 'min_years_experience': 5, 'max_years_experience': 9, 'years_experience_flexible': True, 'target_locations': ['Pune', 'Noida'], 'relocation_provided': True, 'core_skills_required': ['production experience with embeddings-based retrieval systems', 'production experience with vector databases or hybrid search infrastructure', 'strong Python', 'hands-on experience designing evaluation frameworks for ranking systems'], 'work_ethics_and_behavior': ['scrappy product-engineering attitude', 'comfortable with two things that sound contradictory: deep technical depth in modern ML systems and scrappy product-engineering attitude'], 'preferred_past_projects': ['candidate-JD matching at scale', 'recruiter-experience PM collaboration'], 'disqualified_titles': ['Researcher', 'Architect', 'Tech Lead'], 'disqualified_backgrounds': ['academic labs', 'research-only roles', 'closed-source proprietary systems for 5+ years without external validation'], 'additional_note': 'The ideal candidate has a mix of applied ML/AI experience at product companies, has shipped end-to-end systems to real users, and has strong opinions about key areas in AI and ML. They are located in or willing to relocate to Noida or Pune.'}

jd_text = f"""
{jd_schema['role_title']}

{" ".join(jd_schema['target_locations'])}

{"\n".join(jd_schema['core_skills_required'])}

{"\n".join(jd_schema['preferred_past_projects'])}

{jd_schema['additional_note']}
""".strip()

jd_skills = "\n".join(jd_schema['core_skills_required'])

In [5]:
jd_text_embeddings = model.encode(jd_text).tolist()
jd_skills_embeddings = model.encode(jd_skills).tolist()

In [6]:
def candidate_to_text(candidate):
    headline = candidate["profile"]["headline"]
    current_title = candidate["profile"]["current_title"]
    summary = candidate["profile"]["summary"]
    location = candidate["profile"]["location"]
    
    skills = "\n".join(
        skill["name"] if (skill["name"] == "advanced" or skill["name"] == "expert") else ""
        for skill in candidate["skills"]
    )
    
    career = "\n".join(
        job["description"]
        for job in candidate["career_history"]
    )

    candidate_text =  f"""
    {headline}
    {current_title}
    {location}

    {skills}

    {career}
    """.strip()

    return candidate_text, career


In [7]:
def process(filepath):
    print("Spinning up GPUs")
    pool = model.start_multi_process_pool(target_devices=["cuda:0", "cuda:1"])

    texts = []
    careers = []
    records_to_insert = []

    print("Loading data from JSONL")
    with open(filepath, "r") as file:
        for line in file:
            record = json.loads(line)
            text, career = candidate_to_text(record)
            
            texts.append(text)
            careers.append(career)
            
            records_to_insert.append({
                "candidate_id": record['candidate_id']
            })

    print(f"Encoding {len(texts)} resumes.")
    text_embeddings = model.encode(texts, pool=pool, show_progress_bar=True)
    career_embeddings = model.encode(careers, pool=pool, show_progress_bar=True)

    model.stop_multi_process_pool(pool)
    print(f"Generated embeddings shape: {text_embeddings.shape}")

    print("Preparing LanceDB ingestion")
    for i in range(len(records_to_insert)):
        records_to_insert[i]["text_vector"] = text_embeddings[i].tolist()
        records_to_insert[i]["career_desc_vector"] = career_embeddings[i].tolist()
        
    return records_to_insert


In [8]:
embeddings = process("/kaggle/input/datasets/midnightfirefly/redrob-ai-track-1-candidate-resume/dataset/candidates.jsonl")

Spinning up GPUs
Loading data from JSONL
Encoding 100000 resumes.


Chunks:   0%|          | 0/20 [00:00<?, ?it/s]

Chunks:   0%|          | 0/20 [00:00<?, ?it/s]

Generated embeddings shape: (100000, 768)
Preparing LanceDB ingestion


In [9]:
db = lancedb.connect("./lance.db")
db.create_table("candidate_resumes", data=embeddings, mode="overwrite")

[2026-06-22T15:17:27Z WARN  lance::dataset::write::insert] No existing dataset at /kaggle/working/lance.db/candidate_resumes.lance, it will be created


LanceTable(name='candidate_resumes', _conn=LanceDBConnection(uri='/kaggle/working/lance.db'))

In [11]:
db.create_table("jd_text", data=[{"vector": jd_text_embeddings}], mode="overwrite")
db.create_table("jd_skills", data=[{"vector": jd_skills_embeddings}], mode="overwrite")

[2026-06-22T15:17:59Z WARN  lance::dataset::write::insert] No existing dataset at /kaggle/working/lance.db/jd_text.lance, it will be created
[2026-06-22T15:17:59Z WARN  lance::dataset::write::insert] No existing dataset at /kaggle/working/lance.db/jd_skills.lance, it will be created


LanceTable(name='jd_skills', _conn=LanceDBConnection(uri='/kaggle/working/lance.db'))

In [14]:
from IPython.display import FileLink
import os
import shutil

In [15]:
folder_path = '/kaggle/working/lance.db'
shutil.make_archive('output_zip', 'zip', folder_path)
FileLink('output_zip.zip')

/kaggle/working/output_zip.zip